In [ ]:
# INSTALL ======================================================================

! pip install pyTelegramBotAPI schedule pytz

import telebot
from telebot import types
import json
import os
import random
import threading
import time
import math
import schedule
import pytz

from datetime import datetime


TOKEN = "8791421129:AAFcHyaJJvDTob6DGtaJdcyUNTQr2_GCWv0"

bot = telebot.TeleBot(TOKEN)

DATA_FILE = "finance_data.json"

KZ_TIMEZONE = pytz.timezone("Asia/Almaty")


budget_flow = {}
todo_flow = {}


# CATEGORIES ===================================================================
CATEGORIES = [
    "🎭 Entertainment",
    "☕ Cafe & Food",
    "🛒 Shopping",
    "🚌 Transport",
    "🏠 Bills",
    "⚙️ Other"
]


# DATA ENGINE ==================================================================
def load_data():

    if not os.path.exists(DATA_FILE):
        return {}

    try:
        with open(DATA_FILE, "r", encoding="utf-8") as f:
            return json.load(f)

    except:
        return {}

def save_data(data):

    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def get_user_profile(data, user_id):

    uid = str(user_id)

    if uid not in data:

        data[uid] = {

            "settings": {
                "budget_limit": 0
            },

            "transactions": [],

            "todos": []
        }

    return data, uid



# NUMBER CLEANER ===============================================================
def clean_number(text):

    return float(
        text.replace(" ", "").replace(",", ".")
    )



# ADVICE ====================================================================
def get_ai_advice(uid_data, current_category=None):

    transactions = uid_data.get("transactions", [])

    if not transactions:

        return (
            "💡 Start tracking your expenses "
            "to receive personalized financial advice."
        )

    shopping_total = sum(

        t["amount"]

        for t in transactions

        if t.get("category") == "🛒 Shopping"
    )

    entertainment_total = sum(

        t["amount"]

        for t in transactions

        if t.get("category") == "🎭 Entertainment"
    )

    if (
        current_category == "🛒 Shopping"
        and shopping_total > 50000
    ):

        return (
            "🛍 Your shopping expenses are becoming high. "
            "Consider setting a shopping limit."
        )

    if (
        current_category == "🎭 Entertainment"
        and entertainment_total > 40000
    ):

        return (
            "🎉 Entertainment spending is increasing. "
            "Try balancing fun and savings."
        )

    tips = [

        "💰 Saving at least 20% of your income is considered a healthy financial habit.",

        "🏦 Kazakhstan currently has high-interest deposit opportunities compared to many countries.",

        "📈 The 50/30/20 rule helps maintain financial balance.",

        "💡 Small daily expenses can become large yearly costs.",

        "📊 Tracking your expenses regularly improves financial discipline."

    ]

    return random.choice(tips)


# KEYBOARDS ====================================================================
def main_menu():

    markup = types.ReplyKeyboardMarkup(
        resize_keyboard=True,
        row_width=2
    )

    markup.add(
        "💸 Add Expense",
        "💰 Add Income"
    )

    markup.add(
        "📊 Statistics",
        "🧠 Budget Plan"
    )

    markup.add(
        "🏠 Mortgage Calc",
        "📅 Daily Summary"
    )

    markup.add(
        "📝 To-Do Reminder"
    )

    return markup

def category_keyboard():

    markup = types.ReplyKeyboardMarkup(
        resize_keyboard=True,
        one_time_keyboard=True
    )

    btns = [

        types.KeyboardButton(cat)

        for cat in CATEGORIES
    ]

    markup.add(*btns)

    return markup




# START ========================================================================
@bot.message_handler(commands=['start'])
def start(message):

    welcome = (

        "✨ Welcome to FinMate AI ✨\n\n"

        "Your smart financial assistant.\n\n"

        "Functions:\n"

        "💸 Track expenses\n"
        "💰 Add income\n"
        "📊 Financial statistics\n"
        "🧠 Smart budget planner\n"
        "🏠 Mortgage calculator\n"
        "📅 Daily financial summary\n"
        "📝 To-do reminders with notifications\n\n"

        "All transactions are saved with date and time.\n\n"

        "Choose an option below:"
    )

    bot.send_message(
        message.chat.id,
        welcome,
        reply_markup=main_menu()
    )



# INCOME =======================================================================
@bot.message_handler(func=lambda m: m.text == "💰 Add Income")
def income_init(message):

    msg = bot.send_message(
        message.chat.id,
        "💵 Enter income amount:"
    )

    bot.register_next_step_handler(
        msg,
        save_income
    )

def save_income(message):

    try:

        amount = clean_number(message.text)

        data = load_data()

        data, uid = get_user_profile(
            data,
            message.chat.id
        )

        now = datetime.now(KZ_TIMEZONE).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        data[uid]["transactions"].append({

            "type": "income",
            "amount": amount,
            "timestamp": now
        })

        save_data(data)

        bot.send_message(

            message.chat.id,

            f"✅ Income added successfully!\n\n"
            f"💰 Amount: {amount}\n"
            f"🕒 Time: {now}",

            reply_markup=main_menu()
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number. Try again:"
        )

        bot.register_next_step_handler(
            msg,
            save_income
        )



# EXPENSE ======================================================================
@bot.message_handler(func=lambda m: m.text == "💸 Add Expense")
def expense_init(message):

    bot.send_message(

        message.chat.id,

        "📂 Select a category:",

        reply_markup=category_keyboard()
    )

@bot.message_handler(func=lambda m: m.text in CATEGORIES)
def expense_amount_step(message):

    category = message.text

    msg = bot.send_message(

        message.chat.id,

        f"How much did you spend on {category}?"
    )

    bot.register_next_step_handler(
        msg,
        save_expense,
        category
    )

def save_expense(message, category):

    try:

        amount = clean_number(message.text)

        data = load_data()

        data, uid = get_user_profile(
            data,
            message.chat.id
        )

        now = datetime.now(KZ_TIMEZONE).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        data[uid]["transactions"].append({

            "type": "expense",
            "amount": amount,
            "category": category,
            "timestamp": now
        })

        save_data(data)

        advice = get_ai_advice(
            data[uid],
            category
        )

        bot.send_message(

            message.chat.id,

            f"✅ Expense recorded successfully!\n\n"
            f"📂 Category: {category}\n"
            f"💸 Amount: {amount}\n"
            f"🕒 Time: {now}\n\n"
            f"🤖 AI Advice:\n{advice}",

            reply_markup=main_menu()
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number. Try again:"
        )

        bot.register_next_step_handler(
            msg,
            save_expense,
            category
        )



# STATISTICS ===================================================================
@bot.message_handler(func=lambda m: m.text == "📊 Statistics")
def statistics(message):

    data = load_data()

    data, uid = get_user_profile(
        data,
        message.chat.id
    )

    trans = data[uid]["transactions"]

    total_income = sum(
        t["amount"]

        for t in trans

        if t["type"] == "income"
    )

    total_expense = sum(
        t["amount"]

        for t in trans

        if t["type"] == "expense"
    )

    balance = total_income - total_expense

    bot.send_message(

        message.chat.id,

        f"📊 Statistics\n\n"
        f"💰 Total Income: {total_income}\n"
        f"💸 Total Expenses: {total_expense}\n"
        f"📈 Balance: {balance}"
    )



# DAILY SUMMARY ================================================================
@bot.message_handler(func=lambda m: m.text == "📅 Daily Summary")
def daily_summary(message):

    data = load_data()

    data, uid = get_user_profile(
        data,
        message.chat.id
    )

    today = datetime.now(KZ_TIMEZONE).strftime(
        "%Y-%m-%d"
    )

    daily = [

        t

        for t in data[uid]["transactions"]

        if t["timestamp"].startswith(today)
    ]

    spent = sum(

        t["amount"]

        for t in daily

        if t["type"] == "expense"
    )

    advice = get_ai_advice(data[uid])

    bot.send_message(

        message.chat.id,

        f"📅 Daily Summary\n\n"
        f"💸 Spent Today: {spent}\n"
        f"📊 Transactions: {len(daily)}\n\n"
        f"🤖 Advice:\n{advice}"
    )


# MORTGAGE CALCULATOR ==========================================================
@bot.message_handler(func=lambda m: m.text == "🏠 Mortgage Calc")
def mortgage_init(message):

    msg = bot.send_message(

        message.chat.id,

        "Enter:\n"
        "Amount Interest Years\n\n"
        "Example:\n"
        "20000000 14.5 20"
    )

    bot.register_next_step_handler(
        msg,
        mortgage_result
    )

def mortgage_result(message):

    try:

        p = message.text.split()

        amount = float(p[0])

        rate = float(p[1]) / 100 / 12

        years = int(p[2]) * 12

        monthly = (

            amount *

            (rate * (1 + rate) ** years)

            /

            ((1 + rate) ** years - 1)
        )

        total = monthly * years

        overpay = total - amount

        bot.send_message(

            message.chat.id,

            f"🏠 Mortgage Result\n\n"
            f"💵 Monthly Payment: {round(monthly, 2)}\n"
            f"📈 Total Payment: {round(total, 2)}\n"
            f"💸 Interest Paid: {round(overpay, 2)}",

            reply_markup=main_menu()
        )

    except:

        bot.send_message(
            message.chat.id,
            "❌ Invalid format."
        )



# SMART BUDGET PLANNER =========================================================
@bot.message_handler(func=lambda m: m.text == "🧠 Budget Plan")
def budget_start(message):

    msg = bot.send_message(
        message.chat.id,
        "💰 Enter monthly income:"
    )

    bot.register_next_step_handler(
        msg,
        budget_income
    )

def budget_income(message):

    try:

        income = clean_number(message.text)

        budget_flow[message.chat.id] = {
            "income": income
        }

        msg = bot.send_message(
            message.chat.id,
            "🏠 Enter rent/mortgage:"
        )

        bot.register_next_step_handler(
            msg,
            budget_rent
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number. Try again:"
        )

        bot.register_next_step_handler(
            msg,
            budget_income
        )

def budget_rent(message):

    try:

        budget_flow[message.chat.id]["rent"] = clean_number(message.text)

        msg = bot.send_message(
            message.chat.id,
            "💡 Enter utilities:"
        )

        bot.register_next_step_handler(
            msg,
            budget_utilities
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number."
        )

        bot.register_next_step_handler(
            msg,
            budget_rent
        )

def budget_utilities(message):

    try:

        budget_flow[message.chat.id]["utilities"] = clean_number(message.text)

        msg = bot.send_message(
            message.chat.id,
            "🚕 Enter transport costs:"
        )

        bot.register_next_step_handler(
            msg,
            budget_other
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number."
        )

        bot.register_next_step_handler(
            msg,
            budget_utilities
        )

def budget_other(message):

    try:

        budget_flow[message.chat.id]["transport"] = clean_number(message.text)

        msg = bot.send_message(
            message.chat.id,
            "📦 Enter other fixed costs:"
        )

        bot.register_next_step_handler(
            msg,
            budget_result
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number."
        )

        bot.register_next_step_handler(
            msg,
            budget_other
        )

def budget_result(message):

    try:

        other = clean_number(message.text)

        d = budget_flow[message.chat.id]

        income = d["income"]

        fixed = (
            d["rent"] +
            d["utilities"] +
            d["transport"] +
            other
        )

        remaining = income - fixed

        spending = remaining * 0.4
        entertainment = remaining * 0.2
        savings = remaining * 0.2
        deposit = remaining * 0.2

        result = (

            "🧠 SMART BUDGET PLAN\n\n"

            f"💰 Income: {income}\n"
            f"📌 Fixed Costs: {fixed}\n"
            f"📈 Remaining: {remaining}\n\n"

            f"🛒 Spending: {spending:.0f}\n"
            f"🎉 Entertainment: {entertainment:.0f}\n"
            f"💾 Savings: {savings:.0f}\n"
            f"🏦 Deposit: {deposit:.0f}"
        )

        bot.send_message(

            message.chat.id,

            result,

            reply_markup=main_menu()
        )

    except:

        msg = bot.send_message(
            message.chat.id,
            "❌ Invalid number."
        )

        bot.register_next_step_handler(
            msg,
            budget_result
        )



# TO-DO REMINDER SYSTEM ========================================================
@bot.message_handler(func=lambda m: m.text == "📝 To-Do Reminder")
def todo_start(message):

    msg = bot.send_message(

        message.chat.id,

        "📝 Enter task description:\n\n"
        "Example:\n"
        "Pay tuition fee"
    )

    bot.register_next_step_handler(
        msg,
        todo_deadline
    )

def todo_deadline(message):

    todo_flow[message.chat.id] = {
        "task": message.text
    }

    msg = bot.send_message(

        message.chat.id,

        "📅 Enter deadline date:\n"
        "Format: YYYY-MM-DD\n\n"
        "Example:\n"
        "2026-05-15"
    )

    bot.register_next_step_handler(
        msg,
        todo_time
    )

def todo_time(message):

    todo_flow[message.chat.id]["deadline"] = message.text

    msg = bot.send_message(

        message.chat.id,

        "⏰ Enter reminder time:\n"
        "Format: HH:MM\n\n"
        "Example:\n"
        "09:00"
    )

    bot.register_next_step_handler(
        msg,
        todo_repeat
    )

def todo_repeat(message):

    todo_flow[message.chat.id]["time"] = message.text

    markup = types.ReplyKeyboardMarkup(
        resize_keyboard=True,
        one_time_keyboard=True
    )

    markup.add(
        "Every Day",
        "Every 5 Hours"
    )

    msg = bot.send_message(

        message.chat.id,

        "🔁 Choose reminder frequency:",

        reply_markup=markup
    )

    bot.register_next_step_handler(
        msg,
        save_todo
    )

def save_todo(message):

    repeat = message.text

    flow = todo_flow[message.chat.id]

    data = load_data()

    data, uid = get_user_profile(
        data,
        message.chat.id
    )

    flow["repeat"] = repeat

    flow["completed"] = False

    data[uid]["todos"].append(flow)

    save_data(data)

    bot.send_message(

        message.chat.id,

        "✅ Reminder created successfully!",

        reply_markup=main_menu()
    )



# REMINDER CHECKER =============================================================
def reminder_loop():

    while True:

        data = load_data()

        now = datetime.now(KZ_TIMEZONE)

        current_date = now.strftime("%Y-%m-%d")

        current_time = now.strftime("%H:%M")

        for uid in data:

            todos = data[uid].get("todos", [])

            updated_todos = []

            for todo in todos:

                deadline = todo["deadline"]

                reminder_time = todo["time"]

                if current_date > deadline:
                    continue

                if reminder_time == current_time:

                    bot.send_message(

                        int(uid),

                        f"⏰ Reminder\n\n"
                        f"📝 {todo['task']}\n"
                        f"📅 Deadline: {deadline}"
                    )

                updated_todos.append(todo)

            data[uid]["todos"] = updated_todos

        save_data(data)

        time.sleep(60)



# START REMINDER THREAD ========================================================
threading.Thread(
    target=reminder_loop,
    daemon=True
).start()



# RUN ==========================================================================
print("FinMate AI is running...")

bot.infinity_polling()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 9.9 MB/s eta 0:00:00
FinMate AI is running...
